In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


In [2]:
df = pd.read_csv("Downloads/ACME-HappinessSurvey2020.csv") 
print(df.head())
print(df.shape)


   Y  X1  X2  X3  X4  X5  X6
0  0   3   3   3   4   2   4
1  0   3   2   3   5   4   3
2  1   5   3   3   3   3   5
3  0   5   4   3   3   3   5
4  0   5   4   3   3   3   5
(126, 7)


In [3]:
# df: columns X1..X6 (1-5), Y (0/1)
features = ['X1','X2','X3','X4','X5','X6']
X = df[features]
y = df['Y']


In [4]:
# Check class balance first
print(y.value_counts(normalize=True))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


Y
1    0.547619
0    0.452381
Name: proportion, dtype: float64


In [15]:
# Baseline: logistic regression
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

lr_scores = cross_val_score(lr, X, y, cv=cv)

print("LR:", lr_scores.mean())

LR: 0.5550769230769231


In [16]:
# Accuracy leader: gradient boosting
gb = GradientBoostingClassifier(random_state=42)

gb_scores = cross_val_score(gb, X, y, cv=cv)

print("GBM:", gb_scores.mean())



GBM: 0.6107692307692308


In [20]:
#Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42
)

rf_scores = cross_val_score(
    rf,
    X,
    y,
    cv=cv,
    scoring='accuracy'
)

print("Random Forest CV Scores:", rf_scores)
print("Mean Accuracy:", np.mean(rf_scores))

Random Forest CV Scores: [0.65384615 0.64       0.6        0.6        0.4       ]
Mean Accuracy: 0.5787692307692308


In [19]:
#extra trees classifier
from sklearn.ensemble import ExtraTreesClassifier

et = ExtraTreesClassifier(
    n_estimators=500,
    random_state=42
)

et_scores = cross_val_score(
    et,
    X,
    y,
    cv=cv,
    scoring='accuracy'
)

print("Extra Trees CV Scores:", et_scores)
print("Mean Accuracy:", np.mean(et_scores))

Extra Trees CV Scores: [0.65384615 0.64       0.6        0.56       0.56      ]
Mean Accuracy: 0.6027692307692308


In [9]:
!pip install catboost


   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.1/100.2 MB 1.1 MB/s eta 0:01:32
   ---------------------------------------- 0.2/100.2 MB 1.5 MB/s eta 0:01:06
   ---------------------------------------- 0.5/100.2 MB 3.1 MB/s eta 0:00:32
   ---------------------------------------- 1.1/100.2 MB 5.9 MB/s eta 0:00:17
    --------------------------------------- 2.4/100.2 MB 10.1 MB/s eta 0:00:10
   - -------------------------------------- 4.6/100.2 MB 16.2 MB/s eta 0:00:06
   -- ------------------------------------- 6.2/100.2 MB 19.0 MB/s eta 0:00:05
   --- ------------------------------------ 9.0/100.2 MB 24.1 MB/s eta 0:00:04
   --- ------------------------------------ 9.2/100.2 MB 23.6 MB/s eta 0:00:04
   ---- ----------------------------------- 11.0/100.2 MB 34.4 MB/s eta 0:00:03
   ----- ---------------------------------- 12.7/100.2 MB 38.5 MB/s eta 0:00:03
   ----- ---------------------------------- 14.5/100.2 MB 36.4

In [18]:
#catboost classifier
from catboost import CatBoostClassifier

cb = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    verbose=0,
    random_state=42
)

cb_scores = cross_val_score(
    cb,
    X,
    y,
    cv=cv,
    scoring='accuracy'
)

print("CatBoost CV Scores:", cb_scores)
print("Mean Accuracy:", np.mean(cb_scores))

CatBoost CV Scores: [0.73076923 0.72       0.68       0.52       0.6       ]
Mean Accuracy: 0.6501538461538462


In [17]:
results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Gradient Boosting',
        'Random Forest',
        'Extra Trees',
        'CatBoost'
    ],
    'Accuracy': [
        lr_scores.mean(),
        gb_scores.mean(),
        rf_scores.mean(),
        et_scores.mean(),
        cb_scores.mean()
    ]
})

results.sort_values('Accuracy', ascending=False)

,Model,Accuracy
4,CatBoost,0.650154
1,Gradient Boosting,0.610769
3,Extra Trees,0.602769
2,Random Forest,0.578769
0,Logistic Regression,0.555077


In [11]:
rf.fit(X, y)

importance_df = (
    pd.DataFrame({
        'Feature': X.columns,
        'Importance': rf.feature_importances_
    })
    .sort_values('Importance', ascending=False)
)

print(importance_df)

  Feature  Importance
2      X3    0.186960
1      X2    0.180792
4      X5    0.180622
0      X1    0.163803
3      X4    0.155261
5      X6    0.132562


In [21]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

rfe = RFE(model, n_features_to_select=1)
rfe.fit(X, y)

feature_ranking = pd.DataFrame({
    'Feature': X.columns,
    'Rank': rfe.ranking_
}).sort_values('Rank')

print(feature_ranking)

  Feature  Rank
0      X1     1
4      X5     2
2      X3     3
5      X6     4
1      X2     5
3      X4     6


In [22]:
from itertools import combinations
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

results = []

for k in range(1, len(X.columns)+1):
    for subset in combinations(X.columns, k):

        score = cross_val_score(
            RandomForestClassifier(random_state=42),
            X[list(subset)],
            y,
            cv=cv,
            scoring='accuracy'
        ).mean()

        results.append({
            'Features': subset,
            'Num_Features': k,
            'Accuracy': score
        })

results_df = pd.DataFrame(results)
results_df.sort_values('Accuracy', ascending=False).head(20)

,Features,Num_Features,Accuracy
10,"(X1, X6)",2,0.675077
30,"(X1, X5, X6)",3,0.666462
49,"(X1, X3, X5, X6)",4,0.658154
44,"(X1, X2, X4, X5)",4,0.657846
34,"(X2, X4, X5)",3,0.650154
48,"(X1, X3, X4, X6)",4,0.650154
23,"(X1, X2, X5)",3,0.650154
27,"(X1, X3, X6)",3,0.643077
9,"(X1, X5)",2,0.642462
26,"(X1, X3, X5)",3,0.642154


In [23]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

print(importance)

  Feature  Importance
2      X3    0.197599
1      X2    0.177116
4      X5    0.175312
0      X1    0.172084
3      X4    0.147127
5      X6    0.130761
